# 06 | BornMachine Data Modeling

BornMachine is the project's most complete application model. A data-dependent measurement operator injects ordinary continuous samples into the tensor network:

$$P(x)=\langle state|TN^\dagger M_x TN|state\rangle$$

This notebook inspects the five-segment structure, generates Hermite feature maps, and trains a small density model.


In [1]:
from pathlib import Path
import sys

# This works whether Jupyter starts in the repository root or in notebooks/.
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "tneq_qc").is_dir() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/yuch3n/Documents/Code/Github/tneq-qc


In [2]:
import numpy as np
import torch

from tneq_qc import (
    BackendFactory,
    BornMachine,
    DataGenerator,
    EngineCommon,
    QCTNHelper,
    create_optimizer,
    make_data_fn,
)

torch.manual_seed(7)
np.random.seed(7)

backend = BackendFactory.create_backend("pytorch", device="cpu", dtype="float32")
engine = EngineCommon(backend=backend, strategy="row_priority")


## 1. Build a BornMachine

`graph` describes the central trainable TN. `dim=2` is the physical dimension used by state and measurement components.

`auto_init()` initializes the product state, the trainable TN, and identity measurement placeholders. We enable gradients only for the TN before calling `build()`.


In [3]:
N_QUBITS = 3
PHYS_DIM = 2

graph = QCTNHelper.mps(N_QUBITS, bond_dim=2, phys_dim=PHYS_DIM)
model = BornMachine(graph, PHYS_DIM, backend=backend).auto_init(orthogonal=True)
model._submodules["tn"].requires_grad_(True)
combined = model.build()

print(combined)
print("Measurement core names:", model.mx_core_names)
print("Trainable parameters:", [name for name, _ in combined.named_parameters()])


QCTN(nqubits=3, cores=[state.a(2,), state.b(2,), state.c(2,), tn.a(2, 2, 2, 2), tn.b(2, 2, 2, 2), mx.a(2, 2), mx.b(2, 2), mx.c(2, 2), tn_h.a(2, 2, 2, 2), tn_h.b(2, 2, 2, 2), state_t.a(2,), state_t.b(2,), state_t.c(2,)])
Measurement core names: ['mx.a', 'mx.b', 'mx.c']
Trainable parameters: ['tn.a', 'tn.b']


## 2. Hermite feature maps

`DataGenerator` evaluates a Hermite basis `φ(x)` and constructs the outer product

$$M_x=\phi(x)\phi(x)^\dagger.$$

Input shape is `[batch, features]`. The generator returns one `[batch, K, K]` measurement tensor per feature.


In [4]:
data_gen = DataGenerator(backend, mx_K=PHYS_DIM)
x = np.array([
    [-0.8, 0.0, 0.8],
    [ 0.2, 0.4, 0.6],
], dtype=np.float32)

mx_list, phi_x = data_gen.generate(x, K=PHYS_DIM, ret_type="TNTensor")
print("phi(x) shape:", phi_x.shape)
for index, mx_core in enumerate(mx_list):
    print(f"feature {index}: shape={mx_core.shape}, has_batch={mx_core.has_batch}")


phi(x) shape: torch.Size([2, 3, 2])
feature 0: shape=torch.Size([2, 2, 2]), has_batch=True
feature 1: shape=torch.Size([2, 2, 2]), has_batch=True
feature 2: shape=torch.Size([2, 2, 2]), has_batch=True


In [5]:
# Inject a batch into the readable mx.* cores.
for name, mx_core in zip(model.mx_core_names, mx_list):
    combined[name] = mx_core

probabilities = engine.contract(combined)
print("Batch output shape:", probabilities.shape)
print("Batch output:", probabilities.numpy())


[Compiler] Strategy candidates: ['row_priority'], Testing 1 strategies...
  [row_priority] Compatibility: True
  [row_priority] Estimated cost: 5.00e+05 FLOPs
[Compiler] Selected strategy: row_priority (cost: 5.00e+05)
Batch output shape: torch.Size([2])
Batch output: [3.1333973e-04 1.5462021e-05]


## 3. Automatic data injection

`make_data_fn` generates a batch, computes Mx tensors, and updates the `mx.*` cores in place. Its default samples are uniform on `[-1,1]`; a custom `sample_fn` can represent any distribution.


In [6]:
def sample_two_clusters(batch_size, num_qubits):
    centers = np.where(
        np.random.rand(batch_size, 1) < 0.5,
        -0.6,
        0.6,
    )
    noise = 0.15 * np.random.randn(batch_size, num_qubits)
    return (centers + noise).astype(np.float32)


data_fn = make_data_fn(
    data_gen,
    combined,
    batch_size=64,
    num_qubits=N_QUBITS,
    K=PHYS_DIM,
    sample_fn=sample_two_clusters,
)
data_fn(0)
print("A batch has been injected into the model.")


A batch has been injected into the model.


## 4. Train with negative log likelihood

The BornMachine contraction already represents a probability or expectation, not an amplitude. `NLLLoss` therefore uses the real part of a complex result and does not square it again.

The objective is approximately:

$$-\operatorname{mean}(\log(P(x))).$$


In [7]:
optimizer = create_optimizer(
    "sgdg",
    combined.parameters(),
    backend=backend,
    lr=0.01,
)

loss_history = []
for step in range(1, 101):
    data_fn(step)
    loss_value, grads = engine.contract_for_gradient(
        combined,
        target=1,
        loss="nll",
    )
    optimizer.step(list(grads))
    loss_history.append(float(loss_value))

    if step == 1 or step % 20 == 0:
        print(f"step={step:3d} nll={loss_history[-1]:.8f}")


step=  1 nll=5.65253353
step= 20 nll=3.35444713
step= 40 nll=3.02671933
step= 60 nll=3.02305007
step= 80 nll=3.01425719


step=100 nll=3.00141263


## 5. Custom measurement topology

The default model uses one local `K×K` measurement core per qubit. To make one measurement core span several qubits, provide `mx_graph=` to `BornMachine` or compose the five QCTN segments manually.

Keep the default local structure while learning the framework; it matches the probability and sampling APIs most directly.

### Troubleshooting

- Too few Mx tensors: `x.shape[1]` does not match the number of qubits.
- No trainable parameters: gradients were not enabled on `model._submodules["tn"]` before `build()`.
- Unstable NLL: lower the learning rate, check probability positivity, or try a higher precision and automatic scaling.
- Batch treated as a network edge: request `ret_type="TNTensor"` so `has_batch=True` is preserved.

### Exercises

- Replace the two-cluster distribution with a Gaussian.
- Set `PHYS_DIM=4` and inspect measurement and parameter shapes.
- Compare Adam and SGDG on the same random seed.
